# Creating your own dataset

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [78]:
# !pip install datasets evaluate transformers[sentencepiece]
# !apt install git-lfs  # not needed with modern huggingface_hub push_to_hub

You will need to setup git, adapt your email and name in the following cell.

In [79]:
# Git config not required for push_to_hub() with modern huggingface_hub.
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [80]:
from huggingface_hub import notebook_login

notebook_login()

In [81]:
# !pip install requests

In [82]:
import os
import requests
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
headers = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}
print("Authenticated" if GITHUB_TOKEN else "No token found — using unauthenticated (60 req/hour)")

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url, headers=headers)

Authenticated


In [83]:
response.status_code

403

In [84]:
response.json()

{'message': 'API rate limit exceeded for user ID 13595269. If you reach out to GitHub Support for help, please include the request ID BD48:19CFD8:587A2B8:59DA966:69F5AD11 and timestamp 2026-05-02 07:51:45 UTC. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service)',
 'documentation_url': 'https://docs.github.com/rest/overview/rate-limits-for-the-rest-api',
 'status': '403'}

In [85]:
# headers already defined in the imports cell above.
# To authenticate manually instead of .env, uncomment below:
# import getpass
# GITHUB_TOKEN = getpass.getpass("Enter your GitHub token: ")
# headers = {"Authorization": f"token {GITHUB_TOKEN}"}
# SSH keys only work for Git transport (clone/push), not the REST API.

In [86]:
import json
import os
import time
import threading
import math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

MAX_RETRIES = 15
RETRY_WAIT = 5 * 60  # 5 minutes


class RateLimitCoordinator:
    """One thread handles the rate-limit wait; all others block until it wakes."""
    def __init__(self):
        self._event = threading.Event()
        self._event.set()
        self._lock = threading.Lock()
        self._retries = 0

    def wait(self):
        self._event.wait()

    def handle(self, status_code, context=""):
        with self._lock:
            if not self._event.is_set():
                return  # another thread is already handling it
            self._retries += 1
            if self._retries > MAX_RETRIES:
                raise RuntimeError(
                    f"15 attempts made but failed (last status {status_code}"
                    + (f" — {context}" if context else "") + ")"
                )
            label = f" [{context}]" if context else ""
            print(f"Rate limited{label} (attempt {self._retries}/{MAX_RETRIES}), pausing 5 minutes...")
            self._event.clear()
        time.sleep(RETRY_WAIT)
        self._event.set()


def _fetch_page(args):
    page, base_url, owner, repo, per_page, headers, limiter = args
    query = f"issues?page={page}&per_page={per_page}&state=all"
    url = f"{base_url}/{owner}/{repo}/{query}"
    while True:
        limiter.wait()
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            return page, data if isinstance(data, list) else []
        limiter.handle(response.status_code, f"page {page}")


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    issues_path=Path("."),
    headers=None,
):
    if headers is None:
        headers = {}
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    per_page = 100
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"
    num_workers = os.cpu_count()
    limiter = RateLimitCoordinator()

    args_list = [
        (page, base_url, owner, repo, per_page, headers, limiter)
        for page in range(num_pages)
    ]

    pages_data = {}
    with tqdm(total=num_pages, desc="Fetching pages") as pbar:
        with ThreadPoolExecutor(max_workers=num_workers) as pool:
            futures = {pool.submit(_fetch_page, args): args[0] for args in args_list}
            for future in as_completed(futures):
                page_num, issues = future.result()
                pages_data[page_num] = issues
                pbar.update(1)

    # Flatten in page order; stop at first empty page (GitHub paginates contiguously)
    all_issues = []
    for page in range(num_pages):
        issues = pages_data.get(page, [])
        if not issues:
            break
        all_issues.extend(issues)

    out_path = f"{issues_path}/{repo}-issues.jsonl"
    with open(out_path, "w") as f:
        for issue in all_issues:
            f.write(json.dumps(issue, default=str) + "\n")
    print(f"Downloaded {len(all_issues)} issues for {repo}! Stored at {out_path}")


In [87]:
# Depending on your internet connection, this can take several minutes to run...
fetch_issues(headers=headers)

Fetching pages:   0%|          | 0/100 [00:00<?, ?it/s]

Rate limited [page 2] (attempt 1/15), pausing 5 minutes...
Rate limited [page 15] (attempt 2/15), pausing 5 minutes...
Rate limited [page 18] (attempt 3/15), pausing 5 minutes...
Rate limited [page 12] (attempt 4/15), pausing 5 minutes...
Downloaded 8068 issues for datasets! Stored at ./datasets-issues.jsonl


In [88]:
import pandas as pd
from datasets import Dataset

# pd.read_json with convert_dates=False keeps timestamp fields as plain strings,
# preventing PyArrow from inferring `null` on all-null batches then failing on later batches.
df = pd.read_json("datasets-issues.jsonl", lines=True, convert_dates=False)
issues_dataset = Dataset.from_pandas(df)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'pinned_comment', 'draft', 'pull_request'],
    num_rows: 8068
})

In [89]:
sample = issues_dataset.shuffle(seed=666).select(range(3))

# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

>> URL: https://github.com/huggingface/datasets/issues/7617
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/issues/215
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/pull/2171
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/2171', 'html_url': 'https://github.com/huggingface/datasets/pull/2171', 'diff_url': 'https://github.com/huggingface/datasets/pull/2171.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/2171.patch', 'merged_at': '2021-04-06T16:05:09Z'}



In [90]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

Map:   0%|          | 0/8068 [00:00<?, ? examples/s]

In [91]:
issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

In [92]:
_comments_limiter = RateLimitCoordinator()


def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    while True:
        _comments_limiter.wait()
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            return [r["body"] for r in data] if isinstance(data, list) else []
        _comments_limiter.handle(response.status_code, f"issue {issue_number}")


# Test our function works as expected
get_comments(2792)


["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?",
 'Thanks for the help, @albertvillanova! All tests are passing now.']

In [93]:
import os
from concurrent.futures import ThreadPoolExecutor

num_workers = os.cpu_count()
print(f"Using {num_workers} threads")

def get_comments_batch(batch):
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        results = list(pool.map(get_comments, batch["number"]))
    return {"comments": results}

issues_with_comments_dataset = issues_dataset.map(
    get_comments_batch, batched=True, batch_size=num_workers
)

Using 24 threads


Map:   0%|          | 0/8068 [00:00<?, ? examples/s]

Rate limited [issue 3219] (attempt 1/15), pausing 5 minutes...
Rate limited [issue 3207] (attempt 2/15), pausing 5 minutes...
Rate limited [issue 3207] (attempt 3/15), pausing 5 minutes...
Rate limited [issue 3220] (attempt 4/15), pausing 5 minutes...
Rate limited [issue 3211] (attempt 5/15), pausing 5 minutes...
Rate limited [issue 3207] (attempt 6/15), pausing 5 minutes...
Rate limited [issue 3220] (attempt 7/15), pausing 5 minutes...
Rate limited [issue 3207] (attempt 8/15), pausing 5 minutes...
Rate limited [issue 3204] (attempt 9/15), pausing 5 minutes...
Rate limited [issue 3211] (attempt 10/15), pausing 5 minutes...
Rate limited [issue 3220] (attempt 11/15), pausing 5 minutes...
Rate limited [issue 3220] (attempt 12/15), pausing 5 minutes...


In [94]:
# Already logged in above — no need to call notebook_login() again.

In [95]:
issues_with_comments_dataset.push_to_hub("ulises-c/github-issues")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/ulises-c/github-issues/commit/50da37b1a98b08569198734a85e22cbb8cf52672', commit_message='Upload dataset', commit_description='', oid='50da37b1a98b08569198734a85e22cbb8cf52672', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ulises-c/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ulises-c/github-issues'), pr_revision=None, pr_num=None)

In [96]:
remote_dataset = load_dataset("ulises-c/github-issues", split="train")
remote_dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/39.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8068 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'pinned_comment', 'draft', 'pull_request', 'is_pull_request'],
    num_rows: 8068
})